In [1]:
pacman::p_load(hdf5r,dplyr,ggplot2,Matrix,data.table,ggpubr,future,gplots,parallel,lme4)

In [ ]:
setwd('/downstream_analysis/diseaseDiffCREs/')


# functions:
#### double check for hard coded paths

In [ ]:
### Function to read in cell type specific pseudobulk peak file for cell type of interest
### Split into by disease DFs
### 

get_per_sample_gex_SUMS_disease <- function(celltype, adata, condition, samples, outdir){

    test.data <- Matrix::readMM(sprintf('/peakcalling/%s.mtx.gz',celltype))

    test.gene=read.table(sprintf('/peakcalling/%s.peaks',celltype), stringsAsFactors =  TRUE)
    
    test.cell=read.table(sprintf('/peakcalling/%s_mod.barcodes',celltype))
    test.data_m=Matrix::t(test.data)

    rownames(test.data_m)=test.cell$V1
    colnames(test.data_m)=test.gene$V1
    dim(test.data_m)

    #filter peaks
    colMeans_test <- colMeans(test.data_m)
    df <- data.frame(colMeans_test)
    length(df[df$colMeans_test > median(df$colMeans_test),])
    good_peaks <-subset(df, subset = colMeans_test > median(df$colMeans_test))
    subset_mat_goodpeaks <- test.data_m[,colnames(test.data_m) %in% rownames(good_peaks)]

    gex.counts <- t(subset_mat_goodpeaks)
    
    bcs <- names(Idents(adata)[Idents(adata) == condition])
    prefix <- ''
    #pull out rows of gex.counts where BC Ident matches cell_type
    counts <- gex.counts[,colnames(gex.counts) %in% bcs]
    fin.counts.df <- as.data.frame(counts)
     #print(head(gex_list))

    #export df
    mtx.fp <- file.path(outdir,sprintf('%s_bc_gex_total_counts.txt',condition))
    write.table(fin.counts.df,mtx.fp,sep='\t',quote=FALSE, col.names = TRUE)
}

In [ ]:
## get the sequencing depth of each sample
get_depth <- function(samples, celltype, adata){

    samp_depth<-matrix(nrow=length(samples), ncol=2)
    colnames(samp_depth)<-c('Sample', 'Depth')
    samp_depth[,1]<-samples
    samp_depth<-as.data.frame(samp_depth)
    
    test.data <- Matrix::readMM(sprintf('/peakcalling/%s.mtx.gz',celltype))
    test.gene=read.table(sprintf('/peakcalling/%s.peaks',celltype), stringsAsFactors =  TRUE)
    test.cell=read.table(sprintf('/peakcalling/%s_mod.barcodes',celltype))
    test.data_m=Matrix::t(test.data)

    rownames(test.data_m)=test.cell$V1
    colnames(test.data_m)=test.gene$V1
    
    gex.counts <- t(test.data_m)
    
    #pull out rows of gex.counts where BC Ident matches cell_type
    counts <- gex.counts#[,colnames(gex.counts) %in% bcs]

    #go through samples and calculate sum of gex values
    gex_list <- list()
    for (sample in samples){
        if(length(sample_bcs[[sample]]) < 2)next
        sample_cols <- colnames(counts) %in% sample_bcs[[sample]]
        counts.cut <- counts[,sample_cols]
        r_sum<-rowSums(counts.cut)
        c_sum<-colSums(counts.cut)
        all_sum<-sum(r_sum,c_sum)
        samp_depth$Depth[samp_depth$Sample == sample]<-all_sum        
    }
    
    
    samp_depth$Sample<-samples
    samp_depth$Depth<-strtoi(samp_depth$Depth)
    samp_depth <- na.omit(samp_depth)  
    write.table(x = samp_depth, file = 'Samp_depth.txt', sep="\t", quote = FALSE)
    return(samp_depth)
}

# create psuedobulk matrices for logistic regression

In [ ]:
atac<-readRDS('/data/finalATAC_obj.rds')
atac

In [ ]:
celltypes<- unique(atac$FinalSubtypes)
celltypes

In [ ]:
for (CELL in celltypes){
    
    #### create output directories and subset initial object ####
    #dir.create(paste0(CELL,'_diseaseDiff/'))
    setwd(paste0(CELL,'_diseaseDiff/'))
    
    ct_adata <- subset(atac, subset = FinalSubtypes == CELL)
    diseases<- unique(ct_adata$condition_subtype) #get list of all disease states in object
    write.table(diseases, 'diseases.txt',sep='\n', col.names = FALSE, row.names=FALSE, quote=FALSE)
    
    Idents(ct_adata)<-'condition_subtype'
    
    #### read inputs
    samples<-unique(ct_adata[[]]$library)
    write.table(samples, 'sampleIDs.txt',sep='\n', col.names = FALSE, row.names=FALSE, quote=FALSE)
    
    #Create a list of barcodes for each sample
    samp_bars<-list()
    for (samp in samples){
        samp_md<-subset(ct_adata@meta.data, ct_adata@meta.data$library==samp)
        samp_bars[[samp]]<-rownames(samp_md)
    }
    sample_bcs<-samp_bars

    dir.create("peak_count_matrices/")
    outdir<-'peak_count_matrices/'
    
    for (disease in diseases){
        samples <- unique(ct_adata$nPOD_ID[Idents(ct_adata) == disease])
        get_per_sample_gex_SUMS_disease(CELL, ct_adata, disease, samples, outdir)
    }
    
    ##Collecting info on peak sizes (can just pic one file since all have same peaks)
    df<-read.table('peak_count_matrices/ND_bc_gex_total_counts.txt', sep="\t", header=TRUE)
    peak_list<-rownames(df)
    pls<-str_split_fixed(string = peak_list, pattern = "_", n = 3)
    pls<-as.data.frame(pls)
    #string to integer
    pls[,2]<-strtoi(pls[,2])
    pls[,3]<-strtoi(pls[,3])
    final_peaklist<-data.frame(peak_id=peak_list, chrom=pls[,1], start=pls[,2], end=pls[,3], peak.sizes=abs(pls[,3]-pls[,2]))

    print(hist(final_peaklist$peak.sizes))

    write.table(x = final_peaklist, file = 'Peak_sizes.txt', quote = FALSE, sep="\t")
    
    ## get sample depths
    samps<-sub(pattern = "-", replacement = ".", x = samples)
    samples<-unique(ct_adata[[]]$library)

    get_depth(samples, CELL, ct_adata)
    
    
    ## create CPM files 
    dir.create("peak_count_matrices_CPM/")
    #Loop for all pseudobulked matricies
    final_peaklist<-read.table('Peak_sizes.txt')
    samp_depth<-read.table('Samp_depth.txt')
    
    for (DISEASE in diseases){
        df<-read.table(sprintf('peak_count_matrices/%s_bc_gex_total_counts.txt', DISEASE), sep="\t", header=TRUE,check.names = FALSE)
        colnames(df) <- gsub('X','',colnames(df))
        norm_counts<-matrix(nrow=nrow(df), ncol=ncol(df))
        colnames(norm_counts)<-colnames(df)
        rownames(norm_counts)<-rownames(df)
        nct<-length(diseases)
        for (i in 1:ncol(df)){
            raw.counts<-as.numeric(df[,i])
            sample<-str_sub(colnames(df)[i], end = -20)
            if(!(sample %in% samp_depth$Sample)) next
            depth<-as.numeric(samp_depth$Depth[samp_depth$Sample == sample])
            w_size<-as.numeric(final_peaklist$peak.sizes)
            norm_count<-as.numeric((raw.counts*1e6)/(depth)) 
            norm_counts[,i]<-norm_count
            if (i == 1){
                message(paste0("Starting ", DISEASE, " at ", Sys.time()))
                }
            }
        write.table(norm_counts, sprintf('peak_count_matrices_CPM/%s_bc_gex_norm.txt', DISEASE), sep="\t", quote=FALSE)
        message(paste0("Finished at ", Sys.time()))
    }
    setwd('/downstream_analysis/diseaseDiffCREs/')

}


# Subsample large celltypes

In [ ]:
# cell types to subset 'Acinar_1_2_6','Acinar_3','Ductal'
## this cell types were too large to run at full size

## acinar basal

In [ ]:
acinar_basal <- subset(atac, subset= FinalSubtypes == 'Acinar_1_2_6')
acinar_basal

In [ ]:
downsampled.obj <- acinar_basal[, sample(colnames(acinar_basal), size = 10000, replace=F)]
downsampled.obj

In [ ]:
for (CELL in celltypes){
    
    #### create output directories and subset initial object ####
    dir.create(paste0(CELL,'_diseaseDiff_subsamp/'))
    setwd(paste0(CELL,'_diseaseDiff_subsamp/'))
    
    ct_adata <- subset(downsampled.obj, subset = FinalSubtypes == CELL)
    diseases<- unique(ct_adata$condition_subtype) #scan('/home/egriffin/HPAP/Celltypes.txt', what="", sep="\n")
    write.table(diseases, 'diseases.txt',sep='\n', col.names = FALSE, row.names=FALSE, quote=FALSE)
    
    Idents(ct_adata)<-'condition_subtype'
    
    #### read inputs
    samples<-unique(ct_adata[[]]$library)
    write.table(samples, 'sampleIDs.txt',sep='\n', col.names = FALSE, row.names=FALSE, quote=FALSE)
    
    #Create a list of barcodes for each sample
    samp_bars<-list()
    for (samp in samples){
        samp_md<-subset(ct_adata@meta.data, ct_adata@meta.data$library==samp)
        samp_bars[[samp]]<-rownames(samp_md)
    }
    sample_bcs<-samp_bars

    dir.create("peak_count_matrices/")
    outdir<-'peak_count_matrices/'
    
    for (disease in diseases){
        samples <- unique(ct_adata$nPOD_ID[Idents(ct_adata) == disease])
        get_per_sample_gex_SUMS_disease(CELL, ct_adata, disease, samples, outdir)
    }
    
    ##Collecting info on peak sizes (can just pic one file since all have same peaks)
    df<-read.table('peak_count_matrices/ND_bc_gex_total_counts.txt', sep="\t", header=TRUE)
    peak_list<-rownames(df)
    pls<-str_split_fixed(string = peak_list, pattern = "_", n = 3)
    pls<-as.data.frame(pls)
    #string to integer
    pls[,2]<-strtoi(pls[,2])
    pls[,3]<-strtoi(pls[,3])
    final_peaklist<-data.frame(peak_id=peak_list, chrom=pls[,1], start=pls[,2], end=pls[,3], peak.sizes=abs(pls[,3]-pls[,2]))

    print(hist(final_peaklist$peak.sizes))

    write.table(x = final_peaklist, file = 'Peak_sizes.txt', quote = FALSE, sep="\t")
    
    ## get sample depths
    samps<-sub(pattern = "-", replacement = ".", x = samples)
    samples<-unique(ct_adata[[]]$library)

    get_depth(samples, CELL, ct_adata)
    
    
    ## create CPM files 
    dir.create("peak_count_matrices_CPM/")
    #Loop for all pseudobulked matricies
    final_peaklist<-read.table('Peak_sizes.txt')
    samp_depth<-read.table('Samp_depth.txt')
    
    for (DISEASE in diseases){
        df<-read.table(sprintf('peak_count_matrices/%s_bc_gex_total_counts.txt', DISEASE), sep="\t", header=TRUE,check.names = FALSE)
        colnames(df) <- gsub('X','',colnames(df))
        norm_counts<-matrix(nrow=nrow(df), ncol=ncol(df))
        colnames(norm_counts)<-colnames(df)
        rownames(norm_counts)<-rownames(df)
        nct<-length(diseases)
        for (i in 1:ncol(df)){
            raw.counts<-as.numeric(df[,i])
            sample<-str_sub(colnames(df)[i], end = -20)
            if(!(sample %in% samp_depth$Sample)) next
            depth<-as.numeric(samp_depth$Depth[samp_depth$Sample == sample])
            w_size<-as.numeric(final_peaklist$peak.sizes)
            norm_count<-as.numeric((raw.counts*1e6)/(depth)) 
            norm_counts[,i]<-norm_count
            if (i == 1){
                message(paste0("Starting ", DISEASE, " at ", Sys.time()))
                }
            }
        write.table(norm_counts, sprintf('peak_count_matrices_CPM/%s_bc_gex_norm.txt', DISEASE), sep="\t", quote=FALSE)
        message(paste0("Finished at ", Sys.time()))
    }
    setwd('/downstream_analysis/diseaseDiffCREs/')

}


## acinar signalling 1/ acinar 3

In [ ]:
acinar_signal <- subset(atac, subset= FinalSubtypes == 'Acinar_3')
acinar_signal

In [ ]:
downsampled.obj <- acinar_signal[, sample(colnames(acinar_signal), size = 10000, replace=F)]
downsampled.obj

In [ ]:
for (CELL in celltypes){
    
    #### create output directories and subset initial object ####
    dir.create(paste0(CELL,'_diseaseDiff_subsamp/'))
    setwd(paste0(CELL,'_diseaseDiff_subsamp/'))
    
    ct_adata <- subset(downsampled.obj, subset = FinalSubtypes == CELL)
    diseases<- unique(ct_adata$condition_subtype) #scan('/home/egriffin/HPAP/Celltypes.txt', what="", sep="\n")
    write.table(diseases, 'diseases.txt',sep='\n', col.names = FALSE, row.names=FALSE, quote=FALSE)
    
    Idents(ct_adata)<-'condition_subtype'
    
    #### read inputs
    samples<-unique(ct_adata[[]]$library)
    write.table(samples, 'sampleIDs.txt',sep='\n', col.names = FALSE, row.names=FALSE, quote=FALSE)
    
    #Create a list of barcodes for each sample
    samp_bars<-list()
    for (samp in samples){
        samp_md<-subset(ct_adata@meta.data, ct_adata@meta.data$library==samp)
        samp_bars[[samp]]<-rownames(samp_md)
    }
    sample_bcs<-samp_bars

    dir.create("peak_count_matrices/")
    outdir<-'peak_count_matrices/'
    
    for (disease in diseases){
        samples <- unique(ct_adata$nPOD_ID[Idents(ct_adata) == disease])
        get_per_sample_gex_SUMS_disease(CELL, ct_adata, disease, samples, outdir)
    }
    
    ##Collecting info on peak sizes (can just pic one file since all have same peaks)
    df<-read.table('peak_count_matrices/ND_bc_gex_total_counts.txt', sep="\t", header=TRUE)
    peak_list<-rownames(df)
    pls<-str_split_fixed(string = peak_list, pattern = "_", n = 3)
    pls<-as.data.frame(pls)
    #string to integer
    pls[,2]<-strtoi(pls[,2])
    pls[,3]<-strtoi(pls[,3])
    final_peaklist<-data.frame(peak_id=peak_list, chrom=pls[,1], start=pls[,2], end=pls[,3], peak.sizes=abs(pls[,3]-pls[,2]))

    print(hist(final_peaklist$peak.sizes))

    write.table(x = final_peaklist, file = 'Peak_sizes.txt', quote = FALSE, sep="\t")
    
    ## get sample depths
    samps<-sub(pattern = "-", replacement = ".", x = samples)
    samples<-unique(ct_adata[[]]$library)

    get_depth(samples, CELL, ct_adata)
    
    
    ## create CPM files 
    dir.create("peak_count_matrices_CPM/")
    #Loop for all pseudobulked matricies
    final_peaklist<-read.table('Peak_sizes.txt')
    samp_depth<-read.table('Samp_depth.txt')
    
    for (DISEASE in diseases){
        df<-read.table(sprintf('peak_count_matrices/%s_bc_gex_total_counts.txt', DISEASE), sep="\t", header=TRUE,check.names = FALSE)
        colnames(df) <- gsub('X','',colnames(df))
        norm_counts<-matrix(nrow=nrow(df), ncol=ncol(df))
        colnames(norm_counts)<-colnames(df)
        rownames(norm_counts)<-rownames(df)
        nct<-length(diseases)
        for (i in 1:ncol(df)){
            raw.counts<-as.numeric(df[,i])
            sample<-str_sub(colnames(df)[i], end = -20)
            if(!(sample %in% samp_depth$Sample)) next
            depth<-as.numeric(samp_depth$Depth[samp_depth$Sample == sample])
            w_size<-as.numeric(final_peaklist$peak.sizes)
            norm_count<-as.numeric((raw.counts*1e6)/(depth)) 
            norm_counts[,i]<-norm_count
            if (i == 1){
                message(paste0("Starting ", DISEASE, " at ", Sys.time()))
                }
            }
        write.table(norm_counts, sprintf('peak_count_matrices_CPM/%s_bc_gex_norm.txt', DISEASE), sep="\t", quote=FALSE)
        message(paste0("Finished at ", Sys.time()))
    }
    setwd('/downstream_analysis/diseaseDiffCREs/')

}


## ductal cells

In [ ]:
ductal <- subset(atac, subset= FinalSubtypes == 'Ductal')
ductal

downsampled.obj <- ductal[, sample(colnames(ductal), size = 10000, replace=F)]
downsampled.obj

In [ ]:
for (CELL in celltypes){
    
    #### create output directories and subset initial object ####
    dir.create(paste0(CELL,'_diseaseDiff_subsamp/'))
    setwd(paste0(CELL,'_diseaseDiff_subsamp/'))
    
    ct_adata <- subset(downsampled.obj, subset = FinalSubtypes == CELL)
    diseases<- unique(ct_adata$condition_subtype) #scan('/home/egriffin/HPAP/Celltypes.txt', what="", sep="\n")
    write.table(diseases, 'diseases.txt',sep='\n', col.names = FALSE, row.names=FALSE, quote=FALSE)
    
    Idents(ct_adata)<-'condition_subtype'
    
    #### read inputs
    samples<-unique(ct_adata[[]]$library)
    write.table(samples, 'sampleIDs.txt',sep='\n', col.names = FALSE, row.names=FALSE, quote=FALSE)
    
    #Create a list of barcodes for each sample
    samp_bars<-list()
    for (samp in samples){
        samp_md<-subset(ct_adata@meta.data, ct_adata@meta.data$library==samp)
        samp_bars[[samp]]<-rownames(samp_md)
    }
    sample_bcs<-samp_bars

    dir.create("peak_count_matrices/")
    outdir<-'peak_count_matrices/'
    
    for (disease in diseases){
        samples <- unique(ct_adata$nPOD_ID[Idents(ct_adata) == disease])
        get_per_sample_gex_SUMS_disease(CELL, ct_adata, disease, samples, outdir)
    }
    
    ##Collecting info on peak sizes (can just pic one file since all have same peaks)
    df<-read.table('peak_count_matrices/ND_bc_gex_total_counts.txt', sep="\t", header=TRUE)
    peak_list<-rownames(df)
    pls<-str_split_fixed(string = peak_list, pattern = "_", n = 3)
    pls<-as.data.frame(pls)
    #string to integer
    pls[,2]<-strtoi(pls[,2])
    pls[,3]<-strtoi(pls[,3])
    final_peaklist<-data.frame(peak_id=peak_list, chrom=pls[,1], start=pls[,2], end=pls[,3], peak.sizes=abs(pls[,3]-pls[,2]))

    print(hist(final_peaklist$peak.sizes))

    write.table(x = final_peaklist, file = 'Peak_sizes.txt', quote = FALSE, sep="\t")
    
    ## get sample depths
    samps<-sub(pattern = "-", replacement = ".", x = samples)
    samples<-unique(ct_adata[[]]$library)

    get_depth(samples, CELL, ct_adata)
    
    
    ## create CPM files 
    dir.create("peak_count_matrices_CPM/")
    #Loop for all pseudobulked matricies
    final_peaklist<-read.table('Peak_sizes.txt')
    samp_depth<-read.table('Samp_depth.txt')
    
    for (DISEASE in diseases){
        df<-read.table(sprintf('peak_count_matrices/%s_bc_gex_total_counts.txt', DISEASE), sep="\t", header=TRUE,check.names = FALSE)
        colnames(df) <- gsub('X','',colnames(df))
        norm_counts<-matrix(nrow=nrow(df), ncol=ncol(df))
        colnames(norm_counts)<-colnames(df)
        rownames(norm_counts)<-rownames(df)
        nct<-length(diseases)
        for (i in 1:ncol(df)){
            raw.counts<-as.numeric(df[,i])
            sample<-str_sub(colnames(df)[i], end = -20)
            if(!(sample %in% samp_depth$Sample)) next
            depth<-as.numeric(samp_depth$Depth[samp_depth$Sample == sample])
            w_size<-as.numeric(final_peaklist$peak.sizes)
            norm_count<-as.numeric((raw.counts*1e6)/(depth)) 
            norm_counts[,i]<-norm_count
            if (i == 1){
                message(paste0("Starting ", DISEASE, " at ", Sys.time()))
                }
            }
        write.table(norm_counts, sprintf('peak_count_matrices_CPM/%s_bc_gex_norm.txt', DISEASE), sep="\t", quote=FALSE)
        message(paste0("Finished at ", Sys.time()))
    }
    setwd('/downstream_analysis/diseaseDiffCREs/')

}


# Run logistic regression

In [ ]:
wd_meta='/downstream_analysis/'

meta.data =read.table(paste0(wd_meta,'RNA_metadata.txt'),sep = '\t')

meta.data <- meta.data[colnames(meta.data) %in% c('Sample.ID','Beta_proportion')]
colnames(meta.data) <- c('Sample','Beta_proportion')
#meta.data
metadata<-read.csv('atac.metadata.tsv', sep="\t", header=TRUE)
metadata$Sample<-metadata$nPOD_ID
metadata$bc<-rownames(metadata)

metadata$FRiP<-metadata$frac_reads_in_peaks
metadata$count <- metadata$nCount_ATAC
#metadata$bc_prefix <- metadata$library
metadata <- metadata[,colnames(metadata) %in% c('bc','Sample','FRiP','count','bc_prefix')]
metadata_fin <- inner_join(metadata,meta.data,by='Sample',multiple = "all")
metadata_fin <- unique(metadata_fin)

head(metadata_fin)
dim(metadata_fin)

### Beta using beta prop

In [ ]:
cells <- c('Beta')
diseases<- c('Aab','earlyT1D','lateT1D')
diseases

In [ ]:
for (cell in cells){
    setwd('/downstream_analysis/diseaseDiffCREs/')
    dir.create(paste0('logReg_',cell))
    setwd(paste0('logReg_',cell))
    message(paste0("Starting ",cell," at ", Sys.time()))
    #diseases<-scan('diseases.txt', sep="\n", what="")
    diseases <- diseases[! diseases == 'ND']

    for (DISEASE in diseases){
        message(paste0("Starting ",DISEASE," at ", Sys.time()))
        
        no_disease <- 'ND'
        act_disease <- DISEASE
        dir <- #sprintf(',CELL)
        interest<-read.table(sprintf('/downstream_analysis/diseaseDiffCREs/%s_diseaseDiff/peak_count_matrices/%s_bc_gex_total_counts.txt',cell,act_disease), sep="\t", header=TRUE, check.names = FALSE)
        interest$bc <- rownames(interest)
        # Convert dataframe to data.table
        dt <- as.data.table(interest)
#
        # Function to convert integer values to binary
        to_binary <- function(x) {
          as.integer(x != 0)
        }
        
        # Identify integer columns
        integer_cols <- which(sapply(dt, is.integer))
        
        # Convert integer columns to binary
        dfall <- dt[, (integer_cols) := lapply(.SD, to_binary), .SDcols = integer_cols]
        interest <- as.data.frame(dfall)
        rownames(interest) <- interest$bc
        interest <- subset(interest, select=-c(bc))
        other<-read.table(sprintf('/downstream_analysis/diseaseDiffCREs/%s_diseaseDiff/peak_count_matrices/%s_bc_gex_total_counts.txt',cell,no_disease), sep="\t", header=TRUE, check.names = FALSE)
        other <- Filter(function(x)!all(is.na(x)), other)
        other$bc <- rownames(other)
         # Convert dataframe to data.table
        dt <- as.data.table(other)
#
        # Function to convert integer values to binary
        to_binary <- function(x) {
          as.integer(x != 0)
        }
        
        # Identify integer columns
        integer_cols <- which(sapply(dt, is.integer))
        
        # Convert integer columns to binary
        dfall <- dt[, (integer_cols) := lapply(.SD, to_binary), .SDcols = integer_cols]
        other <- as.data.frame(dfall)   
        rownames(other) <- other$bc
        other <- subset(other, select=-c(bc))
        #Read in inputs
        #Reads in files and gets input formatted
        #Read in inputs
        #Reads in files and gets input formatted
        subinterest<-interest#[rownames(interest) %in% ct_peaks, ]
        subother<-other#[rownames(other) %in% ct_peaks, ]
        dfa<-as.data.frame(t(subinterest))
        dfo<-as.data.frame(t(subother))
        dfa$bc<-rownames(dfa)
        dfo$bc<-rownames(dfo)
        dfa$Disease<-act_disease
        dfo$Disease<-'noDisease'
        dfall<-rbind(dfa,dfo)
        dfall$bc <- as.character(dfall$bc)
        dfall <- arrange(dfall) %>%
              select(bc,Disease,everything())
        # Convert dataframe to data.table
        dt <- as.data.table(dfall)

        # Function to convert integer values to binary
        to_binary <- function(x) {
          as.integer(x != 0)
        }
        
        # Identify integer columns
        integer_cols <- which(sapply(dt, is.integer))
        
        # Convert integer columns to binary
        dfall <- dt[, (integer_cols) := lapply(.SD, to_binary), .SDcols = integer_cols]
        dfall <- as.data.frame(dfall)
        int<-left_join(x = dfall, y=metadata_fin, by = 'bc') 
        int$Disease <- as.factor(int$Disease)
        rownames(int) <- int$bc
        input <- arrange(int) %>%
              select(bc,Disease,Sample,FRiP,count,Beta_proportion,everything())
        samples <- list(unique(input$Sample))
        samps_to_remove <- list()
        for(i in 1:length(samples[[1]])){
            print(i)
            SAMP <- as.character(samples[[1]][i])
            print(SAMP)
            if(table(input$Sample)[[SAMP]] < 10){
                samps_to_remove[[length(samps_to_remove) + 1]] = SAMP
            }else{
                print('more than 10 cells')
            }
        }
                        
        input2 <- input[!input$Sample %in% samps_to_remove,]    
        if(length(unique(input2$Disease)) == 1){
                print('only has ND')
        }else{
           #Runs logistic regression in parallel on 20 cores
           ##### change 7 to how ever many columns of metadata i have #####
           res_list<-mclapply(7:ncol(input2), function(i) {
               model1<-glmer(input2[,i] ~ Disease + scale(Beta_proportion) + scale(FRiP) + scale(count) + (1|Sample),  data = input2, family = binomial, nAGQ=1,control = glmerControl(optimizer = "bobyqa"))
               pvalue<-coef(summary(model1))[,'Pr(>|z|)']['DiseasenoDisease']
               ster<-coef(summary(model1))[,'Std. Error']['DiseasenoDisease']
               est<-coef(summary(model1))[,'Estimate']['DiseasenoDisease']
               peak<-colnames(input2)[i]
               c(peak,pvalue, ster,est)
           }, mc.cores=20)
           res_list <- Filter(function(x) !any(grepl("Error", x)), res_list)

           res1<-matrix(unlist(res_list), ncol=4, byrow= TRUE)
           res1<-as.data.frame(res1)
           colnames(res1)<-c("peak",paste0("pvalue_",DISEASE), "std.error","estimate")

          #Calculates log2foldchange from normalized counts matricies and adds this into results dataframe
          L2FC_list<-mclapply(1:nrow(res1), function(i) {
              peak<-res1$peak[i]
              mean_interest<-mean(as.numeric(interest[peak,]))
              mean_other<-mean(as.numeric(other[peak, ]))
              FC<-(mean_interest/mean_other)
              L2FC<-log2(FC)
              c(peak,L2FC,FC)
          }, mc.cores=20)
          addinL2FC<-matrix(unlist(L2FC_list), ncol=3, byrow=TRUE)
          addinL2FC<-as.data.frame(addinL2FC)
          colnames(addinL2FC)<-c('peak', 'L2FC','FC')
          rownames(addinL2FC)<-addinL2FC$peak
          newdf<-left_join(res1, addinL2FC)
          newdf[,paste0("pvalue_",DISEASE)]<-as.numeric(newdf[,paste0("pvalue_",DISEASE)])
          #Adds in bonferroni corrected padj values as a column
          numpeaks<-nrow(newdf)
            fdf_col <- paste0("fdr_",DISEASE)
          newdf[,fdf_col]<-p.adjust(p = newdf[,paste0("pvalue_",DISEASE)], method = "BH", n = nrow(newdf))
          sorteddf<-arrange(newdf, paste0("pvalue_",DISEASE), desc(L2FC))
          #Writes the final table to my directory (should prob change this if I ever share it)
          write.table(sorteddf, sprintf('%s.LogReg.txt', DISEASE),  sep="\t", quote=FALSE, row.names=FALSE)

          #Prints the number of significantly enriched peaks for each celltype based on padj ≤ 0.1
          numsig<-nrow(sorteddf[sorteddf[,fdf_col] <= 0.1 , ])
          numpeaks<-length(6:ncol(input2))
          message(paste("There were ", numsig, " significantly enriched peaks out of ", numpeaks," total peaks for ", DISEASE, " cells relative to ND."))
          message(paste0("Finishing ",DISEASE," at ", Sys.time()))

        }
    }
        message("Finished ", cell, " ", Sys.time())
  setwd('/downstream_analysis/diseaseDiffCREs/')

}

## All celltypes that were not subsampled

In [ ]:
cells <- c('Alpha','Acinar_4','Macrophage','Tcells','Mast','Quiescent_Stellate','LymphEndo',
           'Activated_Stellate','Endothelial','Schwann','Ductal','Delta')

cells

In [ ]:
diseases<- c('Aab','earlyT1D','lateT1D')
diseases

In [ ]:
for (cell in cells){
    setwd('/downstream_analysis/diseaseDiffCREs/')
    dir.create(paste0('logReg_',cell))
    setwd(paste0('logReg_',cell))
    message(paste0("Starting ",cell," at ", Sys.time()))
    #diseases<-scan('diseases.txt', sep="\n", what="")
    diseases <- diseases[! diseases == 'ND']


    for (DISEASE in diseases){
        message(paste0("Starting ",DISEASE," at ", Sys.time()))
        
        no_disease <- 'ND'
        act_disease <- DISEASE
        dir <- #sprintf(',CELL)
        interest<-read.table(sprintf('/downstream_analysis/diseaseDiffCREs/%s_diseaseDiff/peak_count_matrices/%s_bc_gex_total_counts.txt',cell,act_disease), sep="\t", header=TRUE, check.names = FALSE)
        if(ncol(interest) < 2)next
        interest$bc <- rownames(interest)
        # Convert dataframe to data.table
        dt <- as.data.table(interest)
#
        # Function to convert integer values to binary
        to_binary <- function(x) {
          as.integer(x != 0)
        }
        
        # Identify integer columns
        integer_cols <- which(sapply(dt, is.integer))
        
        # Convert integer columns to binary
        dfall <- dt[, (integer_cols) := lapply(.SD, to_binary), .SDcols = integer_cols]
        interest <- as.data.frame(dfall)
        rownames(interest) <- interest$bc
        interest <- subset(interest, select=-c(bc))
        other<-read.table(sprintf('/downstream_analysis/diseaseDiffCREs/%s_diseaseDiff/peak_count_matrices/%s_bc_gex_total_counts.txt',cell,no_disease), sep="\t", header=TRUE, check.names = FALSE)
        other <- Filter(function(x)!all(is.na(x)), other)
        other$bc <- rownames(other)
         # Convert dataframe to data.table
        dt <- as.data.table(other)
#
        # Function to convert integer values to binary
        to_binary <- function(x) {
          as.integer(x != 0)
        }
        
        # Identify integer columns
        integer_cols <- which(sapply(dt, is.integer))
        
        # Convert integer columns to binary
        dfall <- dt[, (integer_cols) := lapply(.SD, to_binary), .SDcols = integer_cols]
        other <- as.data.frame(dfall)   
        rownames(other) <- other$bc
        other <- subset(other, select=-c(bc))
        #Read in inputs
        #Reads in files and gets input formatted
        #Read in inputs
        #Reads in files and gets input formatted
        subinterest<-interest#[rownames(interest) %in% ct_peaks, ]
        subother<-other#[rownames(other) %in% ct_peaks, ]
        dfa<-as.data.frame(t(subinterest))
        dfo<-as.data.frame(t(subother))
        dfa$bc<-rownames(dfa)
        dfo$bc<-rownames(dfo)
        dfa$Disease<-act_disease
        dfo$Disease<-'noDisease'
        dfall<-rbind(dfa,dfo)
        dfall$bc <- as.character(dfall$bc)
        dfall <- arrange(dfall) %>%
              select(bc,Disease,everything())
        # Convert dataframe to data.table
        dt <- as.data.table(dfall)

        # Function to convert integer values to binary
        to_binary <- function(x) {
          as.integer(x != 0)
        }
        
        # Identify integer columns
        integer_cols <- which(sapply(dt, is.integer))
        
        # Convert integer columns to binary
        dfall <- dt[, (integer_cols) := lapply(.SD, to_binary), .SDcols = integer_cols]
        dfall <- as.data.frame(dfall)
        int<-left_join(x = dfall, y=metadata_fin, by = 'bc') 
        int$Disease <- as.factor(int$Disease)
        rownames(int) <- int$bc
        input <- arrange(int) %>%
              select(bc,Disease,Sample,FRiP,count,everything())
        samples <- list(unique(input$Sample))
        samps_to_remove <- list()
        for(i in 1:length(samples[[1]])){
            print(i)
            SAMP <- as.character(samples[[1]][i])
            print(SAMP)
            if(table(input$Sample)[[SAMP]] < 10){
                samps_to_remove[[length(samps_to_remove) + 1]] = SAMP
            }else{
                print('more than 10 cells')
            }
        }
                        
        input2 <- input[!input$Sample %in% samps_to_remove,]   
        if(nrow(input2) == 0)next
        if(length(unique(input2$Disease)) == 1){
                print('only has ND')
        }else{
           #Runs logistic regression in parallel on 20 cores
           ##### change 7 to how ever many columns of metadata i have #####
           res_list<-mclapply(6:ncol(input2), function(i) {
               model1<-glmer(input2[,i] ~ Disease  + scale(FRiP) + scale(count) + (1|Sample),  data = input2, family = binomial, nAGQ=1,control = glmerControl(optimizer = "bobyqa"))
               pvalue<-coef(summary(model1))[,'Pr(>|z|)']['DiseasenoDisease']
               ster<-coef(summary(model1))[,'Std. Error']['DiseasenoDisease']
               est<-coef(summary(model1))[,'Estimate']['DiseasenoDisease']
               peak<-colnames(input2)[i]
               c(peak,pvalue, ster,est)
           }, mc.cores=20,mc.preschedule=TRUE)
           res_list <- Filter(function(x) !any(grepl("Error", x)), res_list)

           res1<-matrix(unlist(res_list), ncol=4, byrow= TRUE)
           res1<-as.data.frame(res1)
           colnames(res1)<-c("peak",paste0("pvalue_",DISEASE), "std.error","estimate")

          #Calculates log2foldchange from normalized counts matricies and adds this into results dataframe
          L2FC_list<-mclapply(1:nrow(res1), function(i) {
              peak<-res1$peak[i]
              mean_interest<-mean(as.numeric(interest[peak,]))
              mean_other<-mean(as.numeric(other[peak, ]))
              FC<-(mean_interest/mean_other)
              L2FC<-log2(FC)
              c(peak,L2FC,FC)
          }, mc.cores=20,mc.preschedule=TRUE)
          addinL2FC<-matrix(unlist(L2FC_list), ncol=3, byrow=TRUE)
          addinL2FC<-as.data.frame(addinL2FC)
          colnames(addinL2FC)<-c('peak', 'L2FC','FC')
          rownames(addinL2FC)<-addinL2FC$peak
          newdf<-left_join(res1, addinL2FC)
          newdf[,paste0("pvalue_",DISEASE)]<-as.numeric(newdf[,paste0("pvalue_",DISEASE)])
          #Adds in bonferroni corrected padj values as a column
          numpeaks<-nrow(newdf)
            fdf_col <- paste0("fdr_",DISEASE)
          newdf[,fdf_col]<-p.adjust(p = newdf[,paste0("pvalue_",DISEASE)], method = "BH", n = nrow(newdf))
          sorteddf<-arrange(newdf, paste0("pvalue_",DISEASE), desc(L2FC))
          #Writes the final table to my directory (should prob change this if I ever share it)
          write.table(sorteddf, sprintf('%s.LogReg.txt', DISEASE),  sep="\t", quote=FALSE, row.names=FALSE)

          #Prints the number of significantly enriched peaks for each celltype based on padj ≤ 0.1
          numsig<-nrow(sorteddf[sorteddf[,fdf_col] <= 0.1 , ])
          numpeaks<-length(6:ncol(input2))
          message(paste("There were ", numsig, " significantly enriched peaks out of ", numpeaks," total peaks for ", DISEASE, " cells relative to ND."))
          message(paste0("Finishing ",DISEASE," at ", Sys.time()))

        }
    }
        message("Finished ", cell, " ", Sys.time())
    setwd('/downstream_analysis/diseaseDiffCREs/')

}

# subsampled large cell types

In [ ]:
cells <- c('Acinar_3','Acinar_1_2_6')
diseases<- c('Aab','earlyT1D','lateT1D')


In [ ]:
for (cell in cells){
    setwd('/downstream_analysis/diseaseDiffCREs/')
    dir.create(paste0('logReg_',cell))
    setwd(paste0('logReg_',cell))
    message(paste0("Starting ",cell," at ", Sys.time()))
    #diseases<-scan('diseases.txt', sep="\n", what="")
    diseases <- diseases[! diseases == 'ND']


    for (DISEASE in diseases){
        message(paste0("Starting ",DISEASE," at ", Sys.time()))
        
        no_disease <- 'ND'
        act_disease <- DISEASE
        dir <- #sprintf(',CELL)
        interest<-read.table(sprintf('/downstream_analysis/diseaseDiffCREs/%s_diseaseDiff_subsamp/peak_count_matrices/%s_bc_gex_total_counts.txt',cell,act_disease), sep="\t", header=TRUE, check.names = FALSE)
        if(ncol(interest) < 2)next
        interest$bc <- rownames(interest)
        # Convert dataframe to data.table
        dt <- as.data.table(interest)
#
        # Function to convert integer values to binary
        to_binary <- function(x) {
          as.integer(x != 0)
        }
        
        # Identify integer columns
        integer_cols <- which(sapply(dt, is.integer))
        
        # Convert integer columns to binary
        dfall <- dt[, (integer_cols) := lapply(.SD, to_binary), .SDcols = integer_cols]
        interest <- as.data.frame(dfall)
        rownames(interest) <- interest$bc
        interest <- subset(interest, select=-c(bc))
        other<-read.table(sprintf('/downstream_analysis/diseaseDiffCREs/%s_diseaseDiff/peak_count_matrices/%s_bc_gex_total_counts.txt',cell,no_disease), sep="\t", header=TRUE, check.names = FALSE)
        other <- Filter(function(x)!all(is.na(x)), other)
        other$bc <- rownames(other)
         # Convert dataframe to data.table
        dt <- as.data.table(other)
#
        # Function to convert integer values to binary
        to_binary <- function(x) {
          as.integer(x != 0)
        }
        
        # Identify integer columns
        integer_cols <- which(sapply(dt, is.integer))
        
        # Convert integer columns to binary
        dfall <- dt[, (integer_cols) := lapply(.SD, to_binary), .SDcols = integer_cols]
        other <- as.data.frame(dfall)   
        rownames(other) <- other$bc
        other <- subset(other, select=-c(bc))
        #Read in inputs
        #Reads in files and gets input formatted
        #Read in inputs
        #Reads in files and gets input formatted
        subinterest<-interest#[rownames(interest) %in% ct_peaks, ]
        subother<-other#[rownames(other) %in% ct_peaks, ]
        dfa<-as.data.frame(t(subinterest))
        dfo<-as.data.frame(t(subother))
        dfa$bc<-rownames(dfa)
        dfo$bc<-rownames(dfo)
        dfa$Disease<-act_disease
        dfo$Disease<-'noDisease'
        dfall<-rbind(dfa,dfo)
        dfall$bc <- as.character(dfall$bc)
        dfall <- arrange(dfall) %>%
              select(bc,Disease,everything())
        # Convert dataframe to data.table
        dt <- as.data.table(dfall)

        # Function to convert integer values to binary
        to_binary <- function(x) {
          as.integer(x != 0)
        }
        
        # Identify integer columns
        integer_cols <- which(sapply(dt, is.integer))
        
        # Convert integer columns to binary
        dfall <- dt[, (integer_cols) := lapply(.SD, to_binary), .SDcols = integer_cols]
        dfall <- as.data.frame(dfall)
        int<-left_join(x = dfall, y=metadata_fin, by = 'bc') 
        int$Disease <- as.factor(int$Disease)
        rownames(int) <- int$bc
        input <- arrange(int) %>%
              select(bc,Disease,Sample,FRiP,count,everything())
        samples <- list(unique(input$Sample))
        samps_to_remove <- list()
        for(i in 1:length(samples[[1]])){
            print(i)
            SAMP <- as.character(samples[[1]][i])
            print(SAMP)
            if(table(input$Sample)[[SAMP]] < 10){
                samps_to_remove[[length(samps_to_remove) + 1]] = SAMP
            }else{
                print('more than 10 cells')
            }
        }
                        
        input2 <- input[!input$Sample %in% samps_to_remove,]   
        if(nrow(input2) == 0)next
        if(length(unique(input2$Disease)) == 1){
                print('only has ND')
        }else{
           #Runs logistic regression in parallel on 20 cores
           ##### change 7 to how ever many columns of metadata i have #####
           res_list<-mclapply(6:ncol(input2), function(i) {
               model1<-glmer(input2[,i] ~ Disease  + scale(FRiP) + scale(count) + (1|Sample),  data = input2, family = binomial, nAGQ=1,control = glmerControl(optimizer = "bobyqa"))
               pvalue<-coef(summary(model1))[,'Pr(>|z|)']['DiseasenoDisease']
               ster<-coef(summary(model1))[,'Std. Error']['DiseasenoDisease']
               est<-coef(summary(model1))[,'Estimate']['DiseasenoDisease']
               peak<-colnames(input2)[i]
               c(peak,pvalue, ster,est)
           }, mc.cores=40,mc.preschedule=TRUE)
           res_list <- Filter(function(x) !any(grepl("Error", x)), res_list)

           res1<-matrix(unlist(res_list), ncol=4, byrow= TRUE)
           res1<-as.data.frame(res1)
           colnames(res1)<-c("peak",paste0("pvalue_",DISEASE), "std.error","estimate")

          #Calculates log2foldchange from normalized counts matricies and adds this into results dataframe
          L2FC_list<-mclapply(1:nrow(res1), function(i) {
              peak<-res1$peak[i]
              mean_interest<-mean(as.numeric(interest[peak,]))
              mean_other<-mean(as.numeric(other[peak, ]))
              FC<-mean_interest/mean_other
              L2FC<-log2(FC)
              c(peak,L2FC,FC)
          }, mc.cores=40,mc.preschedule=TRUE)
          addinL2FC<-matrix(unlist(L2FC_list), ncol=3, byrow=TRUE)
          addinL2FC<-as.data.frame(addinL2FC)
          colnames(addinL2FC)<-c('peak', 'L2FC','FC')
          rownames(addinL2FC)<-addinL2FC$peak
          newdf<-left_join(res1, addinL2FC)
          newdf[,paste0("pvalue_",DISEASE)]<-as.numeric(newdf[,paste0("pvalue_",DISEASE)])
          #Adds in bonferroni corrected padj values as a column
          numpeaks<-nrow(newdf)
            fdf_col <- paste0("fdr_",DISEASE)
          newdf[,fdf_col]<-p.adjust(p = newdf[,paste0("pvalue_",DISEASE)], method = "BH", n = nrow(newdf))
          sorteddf<-arrange(newdf, paste0("pvalue_",DISEASE), desc(L2FC))
          #Writes the final table to my directory (should prob change this if I ever share it)
          write.table(sorteddf, sprintf('%s.LogReg.txt', DISEASE),  sep="\t", quote=FALSE, row.names=FALSE)

          #Prints the number of significantly enriched peaks for each celltype based on padj ≤ 0.1
          numsig<-nrow(sorteddf[sorteddf[,fdf_col] <= 0.1 , ])
          numpeaks<-length(6:ncol(input2))
          message(paste("There were ", numsig, " significantly enriched peaks out of ", numpeaks," total peaks for ", DISEASE, " cells relative to ND."))
          message(paste0("Finishing ",DISEASE," at ", Sys.time()))

        }
    }
        message("Finished ", cell, " ", Sys.time())
    setwd('/downstream_analysis/diseaseDiffCREs/')

}

In [2]:
sessionInfo()

R version 4.2.0 (2022-04-22)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Ubuntu 20.04.2 LTS

Matrix products: default
BLAS/LAPACK: /home/rlmelton/.conda/envs/multiome_fin/lib/libopenblasp-r0.3.21.so

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

attached base packages:
[1] parallel  stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
[1] lme4_1.1-31       gplots_3.1.3      future_1.31.0     ggpubr_0.6.0     
[5] data.table_1.14.8 Matrix_1.5-3      ggplot2_3.4.4     dplyr_1.1.0      
[9] hdf5r_1.3.8      

loaded via a namespace (and not attached):
 [1] Rcpp_1.0.10        lattice_0.20-45    tidyr_1.3.0        listen